# 02 · Spatial autocorrelation and blockCV folds: review before modelling

**Brief sections 7–9.** Run after `R/01_spatial_autocorrelation.R` and `R/02_blockcv_folds.R`. The R outputs are imported here, never regenerated. This notebook:

1. shows the spatial-autocorrelation results: variograms, practical ranges, Moran's I;
2. shows the block size blockCV chose from the autocorrelation range, and validates the outer fold files for that size and its 2× and 3× sensitivity sizes, including a map check that no block holds both training and test boreholes;
3. checks the actual test-to-nearest-training distances, which a block size does not guarantee;
4. checks whether re-using the four remaining outer folds as inner folds gives adequate spatial and class representation;
5. quantifies what a spatial purge would remove;
6. records the block-size decision **before** any model is fitted.

In [ ]:
import os, sys, json
from pathlib import Path

# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner())

from siemreap import data, folds as F, geo
AGG = config.stage_dir(cfg, "04_review", "aggregate")
mt = pd.read_csv(config.processed_dir(cfg) / "cells_all.csv")
sac = config.stage_dir(cfg, "04_spatial_autocorrelation", "aggregate")
bcv = config.stage_dir(cfg, "04_blockcv_folds", "aggregate")

## 1. Spatial autocorrelation (from R)

* **Range parameter vs practical range.** The *range parameter* is a model coefficient. The *practical range* is the distance at which the fitted variogram reaches 95 % of its sill: about 3 × the parameter for an exponential model and √3 × for a Gaussian one. The practical range is the effective correlation distance.
* **Unresolved slices.** `resolved = False` marks slices whose fitted range is shorter than the first lag class (25 m). For those, the variogram is flat at every distance the boreholes sample.
* **Exploratory only.** These are response variograms, which serve as exploratory proxies for residual autocorrelation.

In [ ]:
vp = pd.read_csv(sac / "variogram_parameters.csv"); display(vp)
display(pd.read_csv(sac / "morans_I_by_slice.csv"))
display(pd.read_csv(sac / "blockcv_spatial_autocor_by_slice.csv"))
summary_sac = json.loads((sac / "spatial_autocorrelation_summary.json").read_text())
print(json.dumps(summary_sac, indent=1))
for f in ["fig_variograms_by_slice.png", "fig_pooled_standardised_variogram.png", "fig_vertical_variogram.png"]:
    display(Image(filename=str(sac / f), width=720))

### Block size from blockCV (Valavi et al. 2019)

`blockCV::cv_spatial_autocor` fits a variogram to the Clay indicator of each 1.5 m depth slice and returns the range it recommends as block size (`range_parameter_m`). With several layers, blockCV takes **the median of all calculated ranges**. `R/01_spatial_autocorrelation.R` applies exactly that rule and writes the result to `data/folds/block_size_decision.json`; the folds are built from it. The *practical ranges* of the same variograms are larger, which is why 2× and 3× the size are run as sensitivity analyses.

In [ ]:
b_ac = pd.read_csv(sac / "blockcv_spatial_autocor_by_slice.csv")
display(b_ac[["slice", "model", "range_parameter_m", "practical_range_m", "status"]].round(1))
decision = F.block_decision(cfg)
print(json.dumps(decision, indent=1))
print(f"blockCV block size = median of {decision['slices_used']} slice ranges = {decision['block_size_m']} m")
print("Block sizes of this experiment (primary first):", F.block_sizes(cfg))

## 2. Outer folds: integrity and composition

In [ ]:
blocks = F.block_sizes(cfg)
outer = {b: F.load_outer(cfg, b) for b in blocks}
checks = {b: F.validate_outer(outer[b], mt, cfg) for b in blocks}       # raises on any violation
for b in blocks:
    print(f"{b} m:", {k: v for k, v in checks[b].items() if k != "per_fold"})
    display(pd.DataFrame(checks[b]["per_fold"]).assign(block_m=b))

### Map check: no block holds both training and test boreholes

Each borehole is placed in the R block polygon that contains it. Within every outer fold, a block's boreholes must all be test or all be training; and every borehole's fold must equal its block's fold.

In [ ]:
import geopandas as gpd
bh_xy = mt[mt.in_A].drop_duplicates("BH_Code")[["BH_Code", "UTM_Easting_m", "UTM_Northing_m"]]
pts = gpd.GeoDataFrame(bh_xy, geometry=gpd.points_from_xy(bh_xy.UTM_Easting_m, bh_xy.UTM_Northing_m), crs=cfg["data"]["crs_epsg"])
rows = []
for b in blocks:
    blk = gpd.read_file(config.stage_dir(cfg, "04_blockcv_folds", "restricted") / f"blocks_{b}m.gpkg")
    if "block_id" not in blk:
        blk = blk.reset_index(names="block_id")
    j = gpd.sjoin(pts, blk[["block_id", "folds", "geometry"]], predicate="within", how="left")
    j["fold"] = j.BH_Code.map(outer[b].fold)
    per_block = j.groupby("block_id").fold.nunique()
    rows.append({"block_m": b, "boreholes": len(j), "blocks_with_boreholes": int(per_block.size),
                 "blocks_with_more_than_one_fold": int((per_block > 1).sum()),
                 "borehole_fold_differs_from_block_fold": int((j.fold != j.folds).sum()),
                 "boreholes_outside_blocks": int(j.block_id.isna().sum())})
integrity = pd.DataFrame(rows); display(integrity)
integrity.to_csv(AGG / "block_fold_integrity.csv", index=False)
assert (integrity.blocks_with_more_than_one_fold == 0).all(), "a block holds boreholes of more than one fold"

## 3. Test-to-nearest-training distances (Python check of the R table)

In [ ]:
rows = []
for b in blocks:
    for k in range(1, cfg["blockcv"]["k"] + 1):
        for sc in cfg["experiment"]["scenarios"]:
            tr, te = F.split(mt, outer[b], k, sc)
            d = F.test_train_distances(tr, te)
            rows.append({"block_m": b, "fold": k, "scenario": sc, "test_boreholes": len(d), "min_m": d.min(),
                         "median_m": d.median(), "max_m": d.max(), "share_below_50m": (d < 50).mean(),
                         "share_below_block": (d < b).mean()})
dist_py = pd.DataFrame(rows).round(2)
dist_r = pd.read_csv(bcv / "test_to_train_distance_by_fold.csv")
cmp = dist_py.merge(dist_r, left_on=["block_m", "fold", "scenario"], right_on=["block_size_m", "fold", "scenario"],
                    suffixes=("_py", "_r"))
assert np.allclose(cmp.median_m_py, cmp.median_m_r, atol=0.01), "Python and R distances disagree"
display(dist_py.groupby(["block_m", "scenario"])[["min_m", "median_m", "max_m", "share_below_50m", "share_below_block"]].mean().round(2))
dist_py.to_csv(AGG / "test_to_train_distances.csv", index=False)

A block size does **not** guarantee that size as the minimum separation: a test borehole near a block edge can have a training borehole just across the edge. The `share_below_block` column measures this directly.

## 4. Inner folds: adequacy of re-using the remaining outer folds

In [ ]:
rows = []
for b in blocks:
    for k in range(1, cfg["blockcv"]["k"] + 1):
        for sc in cfg["experiment"]["scenarios"]:
            tr, _ = F.split(mt, outer[b], k, sc)
            g = tr.BH_Code.map(outer[b].fold).astype(int).values.copy(); g[g == 0] = -1
            a = F.adequacy(tr, g, cfg)
            rows.append({"block_m": b, "outer_fold": k, "scenario": sc, "all_ok": bool(a.ok.all()),
                         "min_boreholes": a.boreholes.min(), "min_cells": a.cells.min(),
                         "clay_share_range": f"{a.clay_share.min():.2f}-{a.clay_share.max():.2f}"})
adq = pd.DataFrame(rows); display(adq)
adq.to_csv(AGG / "inner_fold_adequacy.csv", index=False)
print("Outer folds needing the R fallback inner folds:", int((~adq.all_ok).sum()))

## 5. Spatial purge (separate analysis): training boreholes removed

In [ ]:
rows = []
pb = F.primary_block(cfg)
for d in [pb * m for m in cfg["experiment"]["purge_multiples"]]:
    for k in range(1, cfg["blockcv"]["k"] + 1):
        for sc in cfg["experiment"]["scenarios"]:
            tr0, _ = F.split(mt, outer[pb], k, sc)
            tr1, _ = F.split(mt, outer[pb], k, sc, purge_m=d)
            rows.append({"purge_m": d, "fold": k, "scenario": sc, "train_boreholes": tr0.BH_Code.nunique(),
                         "removed": tr0.BH_Code.nunique() - tr1.BH_Code.nunique()})
purge = pd.DataFrame(rows); display(purge.groupby(["purge_m", "scenario"])[["train_boreholes", "removed"]].mean().round(1))
purge.to_csv(AGG / "purge_effect.csv", index=False)

## 6. Block-size decision, recorded before modelling

The primary block size is blockCV's suggestion (section 1); 2× and 3× are sensitivity analyses, and the purge runs remove training boreholes within 1× and 2× of it. `blockCV::cv_distance` compares the test-to-train distances of each design with the distances from the prediction grid to the boreholes (W, lower = closer to how the map is used).

In [ ]:
W = pd.read_csv(bcv / "wasserstein_vs_prediction_domain.csv"); display(W)
for b in blocks:
    f = bcv / f"fig_cv_distance_{b}m.png"
    if f.exists():
        display(Image(filename=str(f), width=560))
record = {"block_sizes_m": blocks, "primary_block_m": F.primary_block(cfg), "decision": decision,
          "purge_distances_m": [F.primary_block(cfg) * m for m in cfg["experiment"]["purge_multiples"]],
          "note": "recorded before any model is fitted (notebook 02)"}
(AGG / "primary_block_decision.json").write_text(json.dumps(record, indent=1, default=str))
print(json.dumps(record, indent=1, default=str))